# 2026-04-28 Generating Boltz-2 input

## Summary

Based on the axial organization, in this notebook we will generate a Boltz-2 input in .yaml file. 

## Dependencies

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import MDAnalysis as mda
import pandas as pd
import yaml
import freesasa
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings
freesasa.setVerbosity(0)

/Users/chloehjj/PCE26/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Functions

In [2]:
def compute_sasa(
    pdb_path: str
) -> dict[str, list[dict]]:
    """Return exposed GLU/ASP and LYS residues sorted by SASA (descending)."""
    structure = freesasa.Structure(pdb_path)
    result = freesasa.calc(structure)
    residue_areas = result.residueAreas()

    out = []
    
    for chain_label, residues in residue_areas.items():
        for res_number, area in residues.items():
            
            entry = {
                "chain": chain_label,
                "residue_number": res_number,
                "residue_type": area.residueType,
                "sasa_total": round(area.total, 2),
                "sasa_side_chain": round(area.sideChain, 2),
                "relative_side_chain": round(area.relativeSideChain, 3),
            }

            out.append(entry)

    return pd.DataFrame.from_records(out)
'''
sasa_df = []
for item in scaffolds:
    tmp_df = compute_sasa('../data/scaffolds/' + item["scaffold"])
    tmp_df['scaffold'] = item['name']
    sasa_df.append(tmp_df)
sasa_df = pd.concat(sasa_df)
sasa_df
'''


'\nsasa_df = []\nfor item in scaffolds:\n    tmp_df = compute_sasa(\'../data/scaffolds/\' + item["scaffold"])\n    tmp_df[\'scaffold\'] = item[\'name\']\n    sasa_df.append(tmp_df)\nsasa_df = pd.concat(sasa_df)\nsasa_df\n'

In [3]:
amino_acid_properties = {
    "ALA": "Nonpolar",
    "VAL": "Nonpolar",
    "LEU": "Nonpolar",
    "ILE": "Nonpolar",
    "PHE": "Nonpolar",
    "TRP": "Nonpolar",
    "MET": "Nonpolar",
    "PRO": "Nonpolar",
    "GLY": "Nonpolar",
    "SER": "Polar",
    "THR": "Polar",
    "CYS": "Polar",
    "TYR": "Polar",
    "ASN": "Polar",
    "GLN": "Polar",
    "LYS": "Basic",
    "ARG": "Basic",
    "HIS": "Basic",
    "ASP": "Acidic",
    "GLU": "Acidic"
}

'''
sasa_df['residue_family'] = sasa_df['residue_type'].map(amino_acid_properties)
'''

"\nsasa_df['residue_family'] = sasa_df['residue_type'].map(amino_acid_properties)\n"

In [4]:
def compute_distance_to_center(
    pdb_path: str
) -> dict[str, list[dict]]:
    """Return exposed GLU/ASP and LYS residues sorted by SASA (descending)."""
    structure = mda.Universe(pdb_path)
    positions = structure.atoms.select_atoms("name CA").positions
    distances = np.linalg.norm(positions, axis=1)
    resnums = structure.atoms.select_atoms("name CA").resnums
    resnames = structure.atoms.select_atoms("name CA").resnames
    chids = structure.atoms.select_atoms("name CA").chainIDs
    u = pd.DataFrame.from_dict(
        dict(chain=chids, residue_number=resnums, residue_type=resnames, distance=distances)
    )
    u.residue_number = u.residue_number.astype(str)
    return u
'''
distances_df = []
for item in scaffolds:
    tmp_df = compute_distance_to_center('../data/scaffolds/' + item["scaffold"])
    tmp_df['scaffold'] = item['name']
    distances_df.append(tmp_df)
distances_df = pd.concat(distances_df)
distances_df
'''

'\ndistances_df = []\nfor item in scaffolds:\n    tmp_df = compute_distance_to_center(\'../data/scaffolds/\' + item["scaffold"])\n    tmp_df[\'scaffold\'] = item[\'name\']\n    distances_df.append(tmp_df)\ndistances_df = pd.concat(distances_df)\ndistances_df\n'